# Experiment runner -- usage example

The road network is loaded once and shared across all experiments.
Each experiment writes its own outputs to `results/<name>/`.

In [1]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

from routing import distrito_tec
from policies.dispatch import HungarianPolicy, GreedyPolicy
from policies.repositioning import StaticPolicy
from run_experiment import ExperimentConfig, run_experiments

# Load once -- shared across all experiments (read-only)
sub_graph, routable_restaurants, residential_zones = distrito_tec()

In [2]:
experiments = [
    ExperimentConfig(
        name             = "hungarian_balanced",
        n_drivers        = 215,
        dispatch_policy  = HungarianPolicy(pickup_radius=3000),
        animate          = True,
        seed             = 42,
        animation_fps= 30
    ),
    ExperimentConfig(
        name             = "greedy_balanced",
        n_drivers        = 215,
        dispatch_policy  = GreedyPolicy(),
        animate          = True,
        seed             = 42,
        animation_fps= 30
    ),
    ExperimentConfig(
        name             = "hungarian_undersupplied",
        n_drivers        = 72,
        dispatch_policy  = HungarianPolicy(pickup_radius=3000),
        animate          = True,   # MP4 only for runs you want to inspect visually
        seed             = 42,
        animation_fps= 30
    ),
    ExperimentConfig(
        name             = "greedy_undersupplied",
        n_drivers        = 72,
        dispatch_policy  = GreedyPolicy(),
        animate          = True,   # MP4 only for runs you want to inspect visually
        seed             = 42,
        animation_fps= 30    
    ),
]

In [ ]:
# n_jobs=1  -> sequential, easier to debug
# n_jobs=2  -> two parallel workers (tune to your core count)
# n_jobs=-1 -> all available cores
results = run_experiments(
    experiments,
    sub_graph,
    routable_restaurants,
    residential_zones,
    n_jobs=-1,
)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.


In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        "name":             r["name"],
        "total_orders":     r["total_orders"],
        "delivered":        r["n_delivered"],
        "avg_e2e_s":        round(r["avg_end_to_end_s"] or 0, 1),
        "avg_dispatch_s":   round(r["avg_dispatch_delay_s"] or 0, 1),
        "avg_food_wait_s":  round(r["avg_food_wait_s"] or 0, 1),
        "wall_time_s":      r["duration_s"],
        "output_dir":       r["output_dir"],
    }
    for r in results
])
summary